In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Examining the distance-per-SoC SoH proxy (feed-only resold Mahindra vehicles)

The 3 resold vehicles are **not** in intellicar (no current → no coulomb counting). The only SoH
method config lists for the Mahindra feed is **distance-per-SoC**: km driven per 1% SoC consumed,
normalized to early life. This notebook tests whether that proxy is usable. Spoiler from the data:
it isn't — and below we show *why* (coarse quantization, dependence on segment size, and no trend
with age), so the conclusion is evidence-based, not asserted.

In [ ]:
import numpy as np, pandas as pd, glob
import plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots

VINS = {'MA1FW2DDUP3A15851':'Zor Grand', 'MA1AG2ZA7R5J80370':'Treo Plus', 'MA1AD2ZA7PJM30725':'Treo Plus'}
rs = pd.read_csv('data/Repo - Pricing - Sold.csv')
rs['reg'] = pd.to_datetime(rs['Registration Date'], format='%d-%b-%Y', errors='coerce')
REG = dict(zip(rs['VIN'], rs['reg']))

def segments(vin):
    d = pd.read_parquet(f'data/mahindra/extra/feed/{vin}.parquet')
    d['t'] = pd.to_datetime(d['t']) if 't' in d.columns else pd.to_datetime(d['eventAt'].astype('int64'), unit='ms')
    for c in ['soc', 'odometer', 'batteryTemp']:
        d[c] = pd.to_numeric(d.get(c), errors='coerce')
    d = d.dropna(subset=['t', 'soc', 'odometer']).sort_values('t').reset_index(drop=True)
    gap = d['t'].diff().dt.total_seconds(); ds = d['soc'].diff(); do = d['odometer'].diff()
    m = (gap > 0) & (gap < 3600) & (ds < -1) & (do > 0) & (do < 200)        # driving + discharging, same trip
    seg = pd.DataFrame({'t': d['t'], 'dsoc': -ds, 'dodo': do,
                        'soc_mid': (d['soc'] + d['soc'].shift()) / 2, 'temp': d['batteryTemp']})[m].copy()
    seg['kmpsoc'] = seg['dodo'] / seg['dsoc']
    seg = seg[(seg['kmpsoc'] > 0.1) & (seg['kmpsoc'] < 5)]
    seg['age_yr'] = (seg['t'] - REG[vin]).dt.days / 365.25
    seg['month'] = seg['t'].dt.to_period('M').dt.to_timestamp()
    seg['vin'] = vin[-6:]; seg['model'] = VINS[vin]
    return seg

SEG = pd.concat([segments(v) for v in VINS], ignore_index=True)
print(f'{len(SEG)} usable discharge segments across {SEG.vin.nunique()} vehicles')
print(SEG.groupby('vin').agg(n=('kmpsoc','size'), dsoc_med=('dsoc','median'),
      kmpsoc_med=('kmpsoc','median'), kmpsoc_cv=('kmpsoc', lambda s: s.std()/s.mean())).round(2).to_string())

## 1. The values are coarsely **quantized** → tiny-segment artifact

Odometer (~0.1–0.5 km resolution) and SoC (~1% steps) are coarse. Most segments span only 1–3% SoC,
so `km/ΔSoC` collapses onto a few ratios (0.5, 1.0, 1.5…). That quantization alone destroys a 1–2%/yr
fade signal.

In [ ]:
fig = px.histogram(SEG, x='kmpsoc', color='vin', nbins=40, barmode='overlay', opacity=0.6,
                   template='plotly_white', title='Distribution of km per %SoC — note the spikes at 0.5 / 1.0 / 1.5 (quantization)')
fig.update_layout(width=1000, height=420, xaxis_title='km per 1% SoC (per discharge segment)', yaxis_title='segments')
frac_round = (np.abs((SEG['kmpsoc'] / 0.5) - (SEG['kmpsoc'] / 0.5).round()) < 0.05).mean()
print(f'{100*frac_round:.0f}% of segment values sit within 5% of a multiple of 0.5 km/%SoC (quantized)')
fig.show()

## 2. The proxy tracks **segment size & driving**, not capacity

`km/ΔSoC` depends on how big the SoC swing was (small swings = quantized/noisy) and on driving
conditions (terrain, load, idle). If it measured capacity it would be flat vs ΔSoC.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('km/%SoC vs segment ΔSoC', 'km/%SoC vs mean SoC of segment'))
for vin, c in zip(SEG.vin.unique(), px.colors.qualitative.Dark24):
    s = SEG[SEG.vin == vin]
    fig.add_scatter(x=s['dsoc'], y=s['kmpsoc'], mode='markers', marker=dict(size=5, color=c, opacity=0.5),
                    name=vin, legendgroup=vin, row=1, col=1)
    fig.add_scatter(x=s['soc_mid'], y=s['kmpsoc'], mode='markers', marker=dict(size=5, color=c, opacity=0.5),
                    name=vin, legendgroup=vin, showlegend=False, row=1, col=2)
fig.update_xaxes(title_text='ΔSoC of segment (%)', row=1, col=1); fig.update_xaxes(title_text='mean SoC (%)', row=1, col=2)
fig.update_yaxes(title_text='km per %SoC', row=1, col=1)
fig.update_layout(width=1050, height=430, template='plotly_white', title_text='km/%SoC is driven by segment size & SoC band, not capacity')
fig.show()

## 3. No degradation **trend with age** — monthly proxy just bounces

If it were a SoH signal the monthly median would decline with age. It doesn't: it rattles around a
flat line, and a linear fit's slope is statistically indistinguishable from zero.

In [ ]:
fig = go.Figure(); print(f"{'vin':8}{'segs/mo':>9}{'slope %/yr':>12}{'monthly CV':>12}")
for vin, c in zip(SEG.vin.unique(), px.colors.qualitative.Dark24):
    s = SEG[SEG.vin == vin]
    mon = s.groupby('month').agg(kmpsoc=('kmpsoc', 'median'), age=('age_yr', 'median'), n=('kmpsoc', 'size'))
    base = mon['kmpsoc'].iloc[:3].median(); mon['proxy_soh'] = (100 * mon['kmpsoc'] / base).clip(upper=100)
    fig.add_scatter(x=mon['age'], y=mon['proxy_soh'], mode='lines+markers', line=dict(color=c), name=vin)
    if len(mon) >= 4:
        sl = np.polyfit(mon['age'], mon['proxy_soh'], 1)[0]
        print(f"{vin:8}{mon['n'].mean():>9.1f}{sl:>12.1f}{(mon['kmpsoc'].std()/mon['kmpsoc'].mean()):>12.2f}")
fig.add_hline(y=80, line=dict(color='red', dash='dot'), annotation_text='80% EoFL')
fig.update_layout(width=1000, height=440, template='plotly_white', xaxis_title='age (years)',
                  yaxis_title='proxy SoH (km/%SoC vs early life, %)', yaxis_range=[40, 105],
                  title_text='"Proxy SoH" over age — bounces 50–100%, no degradation trend (slope ≈ 0)')
fig.show()

## Verdict

The distance-per-SoC proxy is **not a usable SoH signal** for these feed-only vehicles: values are
quantized onto 0.5-km steps, dominated by segment size and driving conditions, and show no trend with
age (slopes ~0, monthly CV is large). A real capacity-fade method needs **current** (coulomb) or
**BMS remaining-capacity** — neither is in the Mahindra feed. So for resold vehicles we report
**usage/mileage** (reliable), not SoH.